# PythonParser Demo

Shows flat and tree parsing of a Python source file using TreeSitter. Flat mode extracts top-level definitions as sequential blocks; tree mode builds a module → class → method hierarchy mirroring the AST.

| Aspect | Notes |
|--------|-------|
| **Pros** | TreeSitter provides exact byte-level node boundaries with error recovery; hierarchy mirrors real Python scope — no heuristics; architecture is language-agnostic so adding TypeScript/Java parsers later is straightforward |
| **Cons** | Requires `tree-sitter` and `tree-sitter-python` extras; does not handle decorators as separate structural nodes; nested classes are flattened to the parent class level |
| **vs. `ast` module** | `ast` is stdlib-only but Python-specific and has no error recovery; TreeSitter is the foundation for multi-language support planned in the roadmap |
| **vs. naive splitting** | Delimiter-based splitting (`def `, `class `) cannot respect scope — a `def` inside a string or comment would be incorrectly treated as a boundary |

## Imports

In [2]:
# Standard Library
import pathlib

# Third Party Library

# Private Library
from cleave.parsers.factory import ParserFactory
from cleave.schemas import Document

## Fixture

In [3]:
FIXTURES = pathlib.Path.cwd().parent.parent / "tests" / "fixtures"
PY_PATH = str(FIXTURES / "sample.py")

if not (FIXTURES / "sample.py").exists():
    from tests.fixtures.py import create_sample_py
    create_sample_py(FIXTURES / "sample.py")

print("PY:", PY_PATH)

PY: c:\Users\SidNa\Documents\GitHub\AI\cleave\notebooks\tests\fixtures\sample.py


| Mode | `Document` field | Use when |
|------|-----------------|----------|
| `flat` | `.pages` — one virtual `DocumentPage`, one block per top-level definition | Fixed or sentence chunking, simple code search |
| `tree` | `.root` — recursive `TreeNode` hierarchy | Semantic retrieval scoped to class/function boundaries |

## Flat mode

One `DocumentPage` with `page_number=None`. The first block is the module docstring (if present), followed by one block per top-level `def` or `class` in source order.

In [7]:
flat_doc_parser = ParserFactory.create(PY_PATH, mode="flat")
flat_doc = flat_doc_parser.parse()
assert isinstance(flat_doc, Document)
print(f"Source type : {flat_doc.source.source_type}")
print(f"Pages       : {len(flat_doc.pages)}")
print(f"Blocks      : {len(flat_doc.pages[0].blocks)}")

Source type : SourceType.python
Pages       : 1
Blocks      : 4


In [8]:
for block in flat_doc.pages[0].blocks:
    preview = block.content[:60].replace("\n", "\\n")
    print(f"  [{block.position}] type={block.type.value:<6}  {preview!r}")

  [0] type=text    'Sample Python module for testing the PythonParser.'
  [1] type=text    'def greet(name: str) -> str:\r\\n    """Return a personalised g'
  [2] type=text    'class DataProcessor:\r\\n    """Process and transform collectio'
  [3] type=text    'async def fetch_data(url: str) -> str:\r\\n    """Simulate asyn'


In [9]:
print("Full text preview:\n")
print(flat_doc.full_text[:400])

Full text preview:

Sample Python module for testing the PythonParser.
def greet(name: str) -> str:
    """Return a personalised greeting string.

    Args:
        name: The name to greet.

    Returns:
        Formatted greeting.
    """
    return f"Hello, {name}!"
class DataProcessor:
    """Process and transform collections of integers."""

    def __init__(self, data: List[int]) -> None:
        ""


## Tree mode

The root `TreeNode` represents the module. Its children are top-level classes and functions; class children are methods. Each node carries `role`, `name`, and `lineno` in its `metadata`.

In [10]:
tree_doc_parser = ParserFactory.create(PY_PATH, mode="tree")
tree_doc = tree_doc_parser.parse()
assert tree_doc.root is not None
print(f"Root role     : {tree_doc.root.metadata['role']}")
print(f"Root name     : {tree_doc.root.metadata['name']}")
print(f"Top children  : {len(tree_doc.root.children)}")

Root role     : module
Root name     : sample.py
Top children  : 3


In [11]:
def print_tree(node, indent=0):
    role = node.metadata.get("role", "")
    name = node.metadata.get("name", "")
    lineno = node.metadata.get("lineno", "")
    tag = f"{role}:{name}" + (f" (L{lineno})" if lineno else "")
    preview = node.content[:50].replace("\n", "\\n")
    print(" " * indent + f"{tag:<35}  {preview!r}")
    for child in node.children:
        print_tree(child, indent + 2)

print_tree(tree_doc.root)

module:sample.py                     'Sample Python module for testing the PythonParser.'
  function:greet (L7)                  'def greet(name: str) -> str:\r\\n    """Return a pers'
  class:DataProcessor (L19)            'Process and transform collections of integers.'
    method:__init__ (L22)                'def __init__(self, data: List[int]) -> None:\r\\n    '
    method:total (L30)                   'def total(self) -> int:\r\\n        """Return the sum'
    method:average (L38)                 'def average(self) -> float:\r\\n        """Return the'
  function:fetch_data (L47)            'async def fetch_data(url: str) -> str:\r\\n    """Sim'


In [ ]:
class_node = next(c for c in tree_doc.root.children if c.metadata.get("role") == "class")
print(f"Class          : {class_node.metadata['name']}")
print(f"Docstring      : {class_node.metadata['docstring']!r}")
print(f"Source preview : {class_node.content[:60]!r}")
print(f"Methods        : {[m.metadata['name'] for m in class_node.children]}")

In [13]:
method = class_node.children[0]
print(f"Method name   : {method.metadata['name']}")
print(f"Role          : {method.metadata['role']}")
print(f"Source:\n{method.content}")

Method name   : __init__
Role          : method
Source:
def __init__(self, data: List[int]) -> None:
        """Initialise with a list of integers.

        Args:
            data: Input data to process.
        """
        self.data = data


## Summary comparison

In [ ]:
flat_text = flat_doc.full_text
tree_text = tree_doc.full_text

print(f"Flat blocks          : {len(flat_doc.pages[0].blocks)}")
print(f"Tree top children    : {len(tree_doc.root.children)}")
print(f"Flat full_text len   : {len(flat_text)}")
print(f"Tree full_text len   : {len(tree_text)}")

func_nodes = [c for c in tree_doc.root.children if c.metadata.get("role") == "function"]
class_nodes = [c for c in tree_doc.root.children if c.metadata.get("role") == "class"]
print(f"\nTop-level functions  : {[n.metadata['name'] for n in func_nodes]}")
print(f"Top-level classes    : {[n.metadata['name'] for n in class_nodes]}")

Flat blocks          : 4
Tree top children    : 3
Flat full_text len   : 1119
Tree full_text len   : 1066

Top-level functions  : ['greet', 'fetch_data']
Top-level classes    : ['DataProcessor']


: 